In [1]:
import os.path
import anndata as ad
import sys
import torch
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from openpyxl import load_workbook
from openpyxl.styles import Alignment, numbers

In [2]:
def save_df(df_results, save_path, drug_name, dataset_name):
    # 按 alpha 列排序
    df_results = df_results.sort_values(by='alpha')

    # 删除重复的 alpha 值，保留第一个出现的值
    df_results = df_results.drop_duplicates(subset='alpha', keep='first')

    # 使用 ExcelWriter 保存文件并应用格式化
    save_file = os.path.join(save_path, f'{drug_name}_{dataset_name}.xlsx')
    with pd.ExcelWriter(save_file, engine='openpyxl') as writer:
        df_results.to_excel(writer, index=False)

        # 获取 workbook 和 worksheet
        workbook = writer.book
        worksheet = writer.sheets['Sheet1']

        # 设置所有列居中对齐
        for row in worksheet.iter_rows():
            for cell in row:
                cell.alignment = Alignment(horizontal='center')

        # 设置 percent_negatives 列为百分比格式
        # 假设 percent_negatives 列是 Excel 中的第4列（D列）
        for cell in worksheet['D']:
            cell.number_format = numbers.FORMAT_PERCENTAGE_00  # 两位小数的百分比格式

        # 自动调整列宽
        for column_cells in worksheet.columns:
            max_length = 0
            column = column_cells[0].column_letter  # 获取列字母
            for cell in column_cells:
                try:
                    if cell.value:
                        max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            adjusted_width = (max_length + 2)  # 增加适当的缓冲宽度
            worksheet.column_dimensions[column].width = adjusted_width

    return

def compute_cost_matrix( sample_tensor, weight, condition=None, source_condition=None):
    if source_condition is not None and condition is not None:
        source = sample_tensor.source[source_condition]
        target = sample_tensor.target[condition].float()
    else:
        source = sample_tensor['source']
        target = sample_tensor['target']

    inner_product_matrix = torch.zeros(source.shape[0], target.shape[0])
    norm_matrix = torch.zeros(source.shape[0], target.shape[0])
    source.requires_grad_(True)

    for j in range(target.shape[0]):
        target_copy = target[j].repeat(source.shape[0], 1)
        norm_matrix[:, j], inner_product_matrix[:, j] = compute_cost(source, target_copy, weight)

    return norm_matrix, inner_product_matrix

def compute_cost(source, target, weight):

    """Compute the cost function for the source and target points.
    Args:
        source (torch.Tensor): Source points.
        target (torch.Tensor): Target points.
        fcn (torch.nn.Module): Function to compute the feature vector.
        alpha (float): Weight for the addition term.
    Returns:
        torch.Tensor: Cost function.
    """
    # Compute the feature vectors for the source and target points.
    with torch.no_grad():
        addition = torch.sum((source @ weight.t()) * (target @ weight.t()), dim=1).squeeze()
        norm = (0.5 * torch.norm(source - target, dim=1)).squeeze()

    return norm, addition


In [3]:
# Init params
adata_path = '/home/jamin/condot/datasets/chemCPA/sciplex_complete_middle_subset_v2.h5ad'
save_path = Path('/home/jamin/condot/results/cost_matrix')
# 创建文件夹（如果不存在）
save_path.mkdir(parents=True, exist_ok=True)

weight_matrix_path = '/home/jamin/condot/notebooks/classifier/classifier_weight_matrix_sciplex.pth'
drug_name = 'all'
dataset_name = adata_path.split('/')[-1].split('.')[0]

# Load classifier
weight_matrix = torch.load(weight_matrix_path, map_location='cuda:0' if torch.cuda.is_available() else 'cpu')['fc.weight']
weight_matrix.requires_grad = False

# Load data
adata = ad.read_h5ad(adata_path)



In [4]:
data = {}
if drug_name != 'all':
    assert drug_name in adata.obs['drug'].unique(), f"Drug name {drug_name} not found in dataset"
    data['source'] = adata[adata.obs['drug'] == 'control'].copy().X.toarray()
    data['target'] = adata[adata.obs['drug'] == drug_name].copy().X.toarray()
else:
    data['source'] = adata[adata.obs['dose'] == 0.0].copy().X.toarray()
    data['target'] = adata[adata.obs['dose'] != 0.0].copy().X.toarray()

data['source'] = torch.tensor(data['source'], device=weight_matrix.device)
data['target'] = torch.tensor(data['target'], device=weight_matrix.device)

if os.path.exists(os.path.join(save_path, f'{drug_name}_{dataset_name}.xlsx')):
    df_results = pd.read_excel(os.path.join(save_path, f'{drug_name}_{dataset_name}.xlsx'))
else:
    # 初始化一个空的 DataFrame，用于保存结果
    df_results = pd.DataFrame(columns=['alpha', 'num_element', 'num_negatives', 'percent_negatives'])

In [5]:
# Compute matrix
norm_matrix, inner_product_matrix = compute_cost_matrix(data, weight_matrix)

num_element = data['source'].shape[0] * data['target'].shape[0]
# Compute cost matrix for different alpha values
for alpha in tqdm(np.concatenate((np.arange(0.0, 0.1, 0.01), np.arange(0.1, 2.0, 0.1), np.arange(2.0, 20.0, 0.5))), desc='Computing cost matrix', ncols=100, position=0, leave=True):

    # Compute cost matrix
    cost_matrix = norm_matrix - (alpha * inner_product_matrix)
    # Compute number of negatives in matrix
    num_negatives = (cost_matrix < 0).sum().item()
    percent_negatives = num_negatives / num_element

    # 将结果保存到 DataFrame 中
    new_row = pd.DataFrame({
        'alpha': [alpha],
        'num_element': [num_element],
        'num_negatives': [num_negatives],
        'percent_negatives': [percent_negatives]
    })
    df_results = pd.concat([df_results, new_row], ignore_index=True)


df_results = df_results.sort_values(by='alpha')

# 保存 DataFrame 为 Excel 文件
save_df(df_results, save_path, drug_name, dataset_name)

Computing cost matrix:   0%|                                                 | 0/65 [00:00<?, ?it/s]/tmp/ipykernel_9798/596316427.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, new_row], ignore_index=True)
Computing cost matrix: 100%|████████████████████████████████████████| 65/65 [17:57<00:00, 16.58s/it]
